# 4. Optional free-T4 micro-fit and gated Gemma comparison

This is an advanced, compute-sensitive demonstration. A five-prompt, four-source-layer fit is useful for learning the mechanics but is **not** a research-quality Jacobian lens. The reference implementation is intentionally not optimized for fitting.

Use notebook 3's public prefit artifact if this runtime is slow or disconnects.

In [ ]:
import sys
from pathlib import Path
repo = Path('/content/slm-jspace-viewer')
if not repo.exists():
    !git clone --branch feature/colab-jspace-tutorial --single-branch https://github.com/prithvimk/slm-jspace-viewer.git {repo}
%cd /content/slm-jspace-viewer
!git fetch origin feature/colab-jspace-tutorial
!git checkout feature/colab-jspace-tutorial
!{sys.executable} -m pip -q install uv
!{sys.executable} -m uv export --locked --no-hashes --no-dev --group tutorial -o /tmp/jspace-colab.txt
!{sys.executable} -m pip -q install -r /tmp/jspace-colab.txt
!{sys.executable} -m pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# The bounded Qwen demonstration. Keep these limits when using a free T4.
import jlens
from tutorials.lib.models import load_tutorial_model
from slm_jspace.selection import jacobian_source_layers

prompts = [
    'The capital of France is Paris.',
    'Two plus two equals four.',
    'A triangle has three sides.',
    'Water freezes at zero degrees Celsius.',
    'The opposite of hot is cold.',
]
tutorial_model = load_tutorial_model()
layer_total = tutorial_model.model.config.num_hidden_layers
source_layers = jacobian_source_layers(layer_total, requested=4)
adapter = jlens.from_hf(tutorial_model.model, tutorial_model.tokenizer)
micro_lens = jlens.fit(adapter, prompts=prompts, source_layers=source_layers, dim_batch=1, max_seq_len=64)
print(f'Fitted demonstration lens over source layers: {source_layers}')

## Optional gated Gemma path

Leave `RUN_GATED_GEMMA` as `False` for the anonymous course. Turning it on requires you to accept Google's Gemma license on Hugging Face and authenticate manually. Do not place tokens in this notebook or commit them to GitHub.

In [ ]:
RUN_GATED_GEMMA = False

if RUN_GATED_GEMMA:
    from huggingface_hub import notebook_login
    from slm_jspace.config import ModelConfig
    from slm_jspace.modeling import load_model
    notebook_login()  # User-controlled authentication after accepting Gemma's model license.
    gemma = load_model(ModelConfig.load('configs/model.gemma-270m.yaml'))
    print(f'Loaded {gemma.config.model_id}. Use a published prefit artifact for comparison; do not expect a full fit to be stable on free Colab.')
else:
    print('Skipped gated Gemma path. The Qwen and public-artifact paths need no login.')

## Takeaway

Small fits teach the estimator's workflow. They should not be used to make strong scientific claims about lens quality, cross-model scaling, or model reasoning. Record model revision, precision, corpus, selected source layers, and package versions for every artifact.